# Sub-ice shelf bathymetry

Floating ice shelves present a challenge to measuring the shape of the seafloor beneath them. Conventional techniques of measuring bathymetry / bed topography, such as shipboard multibeam offshore, or airborne radar onshore, in not possible due to the consistent ice coverage. Inversion of airborne gravity data provides a technique to model the bathymetry in these regions. Here we will demonstrate the gravity reduction and inversion steps unique to a sub-ice shelf bathymetry inversion. We will apply this to a portion of Antarctica's Filchner-Ronne Ice Shelf. 

## Import packages

In [ ]:
# %load_ext autoreload
# %autoreload 2

import os

import harmonica as hm
import numpy as np
import polartoolkit as ptk

import invert4geom

# set EPSG for plotting functions
os.environ["POLARTOOLKIT_EPSG"] = "3031"

## Get gravity data
Here we will load the AntGG-2021 grid of gravity disturbances, an Antarcitc-wide gravity compilation consisting of ground, airborne, and satellite gravity observations. 

In [ ]:
spacing = 2e3

# get region
region = [
    -1625e3,
    -1450e3,
    -750e3,
    -540e3,
]

# add optional buffer
# region = ptk.alter_region(region, zoom=-20e3)

# change to nearest multiple of spacing
region = tuple([spacing * round(x / spacing) for x in region])
region

In [ ]:
grav_data = ptk.fetch.gravity(
    version="antgg-2021",
    anomaly_type="DG",
    region=region,
    spacing=spacing,
)

# rename coordinates to easting and northing
grav_data = grav_data.rename({"x": "easting", "y": "northing"})

# rename elevation variable to upward
grav_data = grav_data.rename({"ellipsoidal_height": "upward"})

# initialize as invert4geom data object
grav_data = invert4geom.create_data(grav_data)
grav_data

In [ ]:
fig = ptk.plot_grid(
    grav_data.gravity_disturbance,
    title="Gravity disturbance",
    cmap="balance+h0",
    cbar_label="mGal",
    hist=True,
    frame=["nSwE", "xaf20000", "yaf20000"],
    coast=True,
    absolute=True,
    robust=True,
)
fig.show()

## Convert FAA to preliminary topography

In [ ]:
gravitational_constant = 6.6743e-11
topo_initial = grav_data.gravity_disturbance / (
    1e5 * 2 * np.pi * gravitational_constant * (2670 - 1028)
)

In [ ]:
topo_bedmap = ptk.fetch.bedmap2(layer="bed", region=region, spacing=spacing)
topo_bedmap = topo_bedmap.rename({"x": "easting", "y": "northing"})

In [ ]:
_ = ptk.grid_compare(
    topo_initial,
    topo_bedmap,
    plot=True,
    hist=True,
)

In [ ]:
fig = ptk.plot_grid(
    topo_initial,
    title="Preliminary topography estimate",
    cmap="globe",
    cbar_label="m",
    hist=True,
    frame=["nSwE", "xaf20000", "yaf20000"],
    coast=True,
    # robust=True,
)
fig.show()

## Isostatic Moho

In [ ]:
moho = -1 * hm.isostatic_moho_airy(
    basement=topo_initial,
    # basement=topo_bedmap,
    reference_depth=30e3,
    density_crust=2670,
    density_mantle=3330,
)

moho = moho.to_dataset(name="upward")

fig = ptk.plot_grid(
    moho.upward,
    title="Airy Moho",
    cmap="broc",
    cbar_label="m",
    hist=True,
    frame=["nSwE", "xaf20000", "yaf20000"],
    coast=True,
    # robust=True,
    cpt_lims=(-32e3, -25e3),
)
fig.show()

## Moho gravity effect

In [ ]:
moho_prisms = invert4geom.create_model(
    zref=-30e3,
    density_contrast=3300 - 2670,
    topography=moho,
)

moho_prisms.inv.plot_model(
    color_by="density",
    zscale=10,
)

In [ ]:
grav_data.inv.forward_gravity(
    layer=moho_prisms,
    name="moho_gravity_effect",
    progressbar=True,
)

In [ ]:
fig = ptk.plot_grid(
    grav_data.moho_gravity_effect,
    title="Moho gravity effect",
    cmap="magma",
    cbar_label="mGal",
    hist=True,
    frame=["nSwE", "xaf20000", "yaf20000"],
    coast=True,
    # robust=True,
    # cpt_lims=(20, 60)
)
fig.show()

## Compare Moho gravity effect with low pass filtering or upward continuation

In [ ]:
filter_faa = -1 * invert4geom.filter_grid(
    grav_data.gravity_disturbance,
    filter_width=150e3,
    filter_type="lowpass",
)

_ = ptk.grid_compare(
    grav_data.moho_gravity_effect,
    filter_faa,
    hist=True,
    coast=True,
)

In [ ]:
filter_widths = np.linspace(50e3, 500e3, 100)

scores = []
for f in filter_widths:
    filtered = -1 * invert4geom.filter_grid(
        grav_data.gravity_disturbance,
        filter_width=f,
        filter_type="lowpass",
    )
    score = invert4geom.rmse(grav_data.moho_gravity_effect - filtered)
    scores.append(score)
print(scores)
print(filter_widths)

best_score = np.min(scores)
print(f"Best score: {best_score}")

best_filter = filter_widths[np.argmin(scores)]
print(f"Best filter: {best_filter}")

filtered = -1 * invert4geom.filter_grid(
    grav_data.gravity_disturbance,
    filter_width=best_filter,
    filter_type="lowpass",
)
_ = ptk.grid_compare(
    grav_data.moho_gravity_effect,
    filtered,
    hist=True,
    coast=True,
)

In [ ]:
filter_heights = np.linspace(5e3, 50e3, 100)

scores = []
for f in filter_heights:
    filtered = -1 * invert4geom.filter_grid(
        grav_data.gravity_disturbance,
        height_displacement=f,
        filter_type="up_continue",
    )
    score = invert4geom.rmse(grav_data.moho_gravity_effect - filtered)
    scores.append(score)
print(scores)
print(filter_heights)

best_score = np.min(scores)
print(f"Best score: {best_score}")

best_filter = filter_heights[np.argmin(scores)]
print(f"Best filter: {best_filter}")

filtered = -1 * invert4geom.filter_grid(
    grav_data.gravity_disturbance,
    height_displacement=best_filter,
    filter_type="up_continue",
)
_ = ptk.grid_compare(
    grav_data.moho_gravity_effect,
    filtered,
    hist=True,
    coast=True,
)

## Isostatic adjusted free-air anomaly

In [ ]:
grav_data["g_iso_adjusted"] = (
    grav_data.gravity_disturbance - grav_data.moho_gravity_effect
)

fig = ptk.plot_grid(
    grav_data.g_iso_adjusted,
    title="Isostatically adjusted Free air anomaly",
    cmap="balance+h0",
    cbar_label="mGal",
    hist=True,
    frame=["nSwE", "xaf20000", "yaf20000"],
    coast=True,
    absolute=True,
    robust=True,
)
fig.show()

In [ ]:
cpt_lims = ptk.get_combined_min_max(
    [grav_data.gravity_disturbance, grav_data.g_iso_adjusted],
    robust=True,
    absolute=True,
)
fig = ptk.subplots(
    [grav_data.gravity_disturbance, grav_data.g_iso_adjusted],
    titles=["FAA", "Isostatic-adjusted FAA"],
    cmap="balance+h0",
    cbar_label="mGal",
    hist=True,
    frame=["nSwE", "xaf20000", "yaf20000"],
    coast=True,
    insets=[True, False],
    absolute=True,
    cpt_lims=cpt_lims,
)

fig.show()

## Initial topo from isostatic FAA

In [ ]:
topo_initial_iso = grav_data.g_iso_adjusted / (
    1e5 * 2 * np.pi * gravitational_constant * (2670 - 1028)
)

fig = ptk.plot_grid(
    topo_initial_iso,
    title="Initial topography estimate from isostatic anomaly",
    cmap="globe",
    cbar_label="m",
    hist=True,
    frame=["nSwE", "xaf20000", "yaf20000"],
    coast=True,
    # robust=True,
)
fig.show()

## Forward gravity of initial topo

In [ ]:
topo_prisms = invert4geom.create_model(
    zref=0,
    density_contrast=2670 - 1028,
    topography=topo_initial_iso.to_dataset(name="upward"),
)

grav_data.inv.forward_gravity(
    layer=topo_prisms,
    name="topo_iso_gravity_effect",
    progressbar=True,
)

In [ ]:
grav_data

In [ ]:
_ = ptk.grid_compare(
    grav_data.g_iso_adjusted,
    grav_data.topo_iso_gravity_effect,
    hist=True,
    coast=True,
)

## Forward gravity of water

In [ ]:
topo_initial_iso.where(topo_initial_iso < 0, 0).plot()

In [ ]:
water_prisms = invert4geom.create_model(
    zref=0,
    density_contrast=-1028,
    topography=topo_initial_iso.where(topo_initial_iso < 0, 0).to_dataset(
        name="upward"
    ),
)

water_prisms.inv.plot_model(
    color_by="thickness",
    zscale=10,
)

In [ ]:
grav_data.inv.forward_gravity(
    layer=water_prisms,
    name="water_gravity_effect",
    progressbar=True,
)

In [ ]:
fig = ptk.plot_grid(
    grav_data.water_gravity_effect,
    title="Water gravity effect",
    cmap="magma",
    cbar_label="mGal",
    hist=True,
    frame=["nSwE", "xaf20000", "yaf20000"],
    coast=True,
    # robust=True,
    # cpt_lims=(20, 60)
)
fig.show()

In [ ]:
_ = ptk.grid_compare(
    grav_data.g_iso_adjusted,
    grav_data.topo_iso_gravity_effect - grav_data.water_gravity_effect,
    hist=True,
    coast=True,
)

In [ ]:
grav_cpt_lims = ptk.get_combined_min_max(
    [grav_data.gravity_disturbance, grav_data.g_iso_adjusted],
    robust=True,
    absolute=True,
)
topo_cpt_lims = ptk.get_combined_min_max(
    [
        topo_initial,
        topo_initial_iso,
    ],
    robust=True,
)
fig = ptk.plot_grid(
    grav_data.gravity_disturbance,
    title="Input free-air anomaly",
    cmap="balance+h0",
    cbar_label="mGal",
    hist=True,
    frame=["nSWe", "xaf20000", "yaf20000"],
    coast=True,
    cpt_lims=grav_cpt_lims,
)
fig = ptk.plot_grid(
    # topo_initial,
    topo_bedmap,
    title="Preliminary topography estimate",
    cmap="globe",
    cbar_label="m",
    hist=True,
    frame=["nSwe", "xaf20000", "yaf20000"],
    coast=True,
    # robust=True,
    cpt_lims=topo_cpt_lims,
    fig=fig,
    origin_shift="x",
)
fig = ptk.plot_grid(
    moho.upward,
    title="Airy Moho",
    cmap="broc",
    cbar_label="m",
    hist=True,
    frame=["nSwE", "xaf20000", "yaf20000"],
    coast=True,
    # robust=True,
    cpt_lims=(-32e3, -25e3),
    fig=fig,
    origin_shift="x",
)
fig = ptk.plot_grid(
    grav_data.moho_gravity_effect,
    title="Moho gravity effect",
    cmap="balance+h0",
    cbar_label="mGal",
    hist=True,
    frame=["nSwE", "xaf20000", "yaf20000"],
    coast=True,
    # robust=True,
    # cpt_lims=(20, 60),
    cpt_lims=grav_cpt_lims,
    fig=fig,
    origin_shift="x",
)

fig = ptk.plot_grid(
    grav_data.g_iso_adjusted,
    title="Isostatically adjusted FAA",
    cmap="balance+h0",
    cbar_label="mGal",
    hist=True,
    frame=["nSwE", "xaf20000", "yaf20000"],
    coast=True,
    absolute=True,
    robust=True,
    fig=fig,
    origin_shift="both",
    xshift_amount=-3,
)
fig = ptk.plot_grid(
    topo_initial_iso,
    title="Topography from isostatic anomaly",
    cmap="globe",
    cbar_label="m",
    hist=True,
    frame=["nSwe", "xaf20000", "yaf20000"],
    coast=True,
    # robust=True,
    cpt_lims=topo_cpt_lims,
    fig=fig,
    origin_shift="x",
)
fig.show()

In [ ]:
data_dict = ptk.make_data_dict(
    ["FAA", "Moho forward gravity", "FAA_iso"],
    [
        grav_data.gravity_disturbance,
        grav_data.moho_gravity_effect,
        grav_data.g_iso_adjusted,
    ],
    ["black", "red", "purple"],
)

layers_dict = ptk.make_data_dict(
    ["Topo from FAA", "Topo from FAA-iso", "moho"],
    [topo_initial, topo_initial_iso, moho.upward],
    ["blue", "red", "black"],
)

fig, _, _ = ptk.plot_profile(
    "points",
    start=(region[0], region[3]),
    stop=(region[1], region[2]),
    layers_dict=layers_dict,
    data_dict=data_dict,
    fill_layers=False,
    fig_width=10,
    fig_height=12,
    data_height=6,
    epsg="3031",
    add_map=True,
)
fig.show()